In [1]:
!python3 -m pip install scikit-learn

In [29]:
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from xgboost import XGBClassifier
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

In [4]:
df = sns.load_dataset('titanic')
X = df[["sex", "age", "fare", "embarked","pclass"]]
y = df["survived"]

In [10]:
model = Pipeline([
    ('preprocessor', ColumnTransformer([
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), ['age', 'fare']),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ]), ['sex', 'embarked', 'pclass'])
    ])),
    ('classifier', RandomForestClassifier())  # placeholder
])

In [11]:
param_grids = [

    # 🌳 Random Forest
    {
        'classifier': [RandomForestClassifier(random_state=42)],
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth': [None, 10, 20],
        'classifier__min_samples_split': [2, 5],
        'classifier__min_samples_leaf': [1, 2]
    },

    # 📈 Logistic Regression
    {
        'classifier': [LogisticRegression(random_state=42, max_iter=1000)],
        'classifier__C': [0.1, 1, 10],
        'classifier__solver': ['liblinear', 'lbfgs']
    },

    # ⚡ XGBoost
    {
        'classifier': [XGBClassifier(eval_metric='logloss', random_state=42)],
        'classifier__learning_rate': [0.01, 0.1],
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth': [3, 6],
        'classifier__min_child_weight': [1, 5]
    }
]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
grid_search = GridSearchCV(model, param_grid=param_grids, cv=cv, n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

Fitting 5 folds for each of 46 candidates, totalling 230 fits
[CV] END classifier=RandomForestClassifier(random_state=42), classifier__max_depth=None, classifier__min_samples_leaf=1, classifier__min_samples_split=2, classifier__n_estimators=100; total time=   0.1s
[CV] END classifier=RandomForestClassifier(random_state=42), classifier__max_depth=None, classifier__min_samples_leaf=1, classifier__min_samples_split=2, classifier__n_estimators=100; total time=   0.1s
[CV] END classifier=RandomForestClassifier(random_state=42), classifier__max_depth=None, classifier__min_samples_leaf=1, classifier__min_samples_split=2, classifier__n_estimators=100; total time=   0.1s
[CV] END classifier=RandomForestClassifier(random_state=42), classifier__max_depth=None, classifier__min_samples_leaf=1, classifier__min_samples_split=2, classifier__n_estimators=100; total time=   0.1s
[CV] END classifier=RandomForestClassifier(random_state=42), classifier__max_depth=None, classifier__min_samples_leaf=1, class

In [22]:
print("Best Hyperparameters:", grid_search.best_params_)
print("Best Model:", best_model.named_steps['classifier'])
print("CV_score:", grid_search.cv_results_['mean_test_score'][grid_search.best_index_])
print("importance:", best_model.named_steps['classifier'].feature_importances_ if hasattr(best_model.named_steps['classifier'], 'feature_importances_') else "N/A")
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))

Best Hyperparameters: {'classifier': XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...), 'classifier__learning_rate': 0.1, 'classifier__max_depth': 6, 'classifier__min_child_weight': 5, 'classifier__n_estimators': 100}
Best Model: XGBClassifier(base_score=None, booster=N

In [20]:
cv_results = pd.DataFrame(grid_search.cv_results_)

summary = cv_results[['param_classifier', 'mean_test_score']]

# Get best row per model
summary['Model'] = summary['param_classifier'].apply(lambda x: type(x).__name__)
summary = summary.sort_values(by='mean_test_score', ascending=False)

print(summary[['Model', 'mean_test_score']])

                     Model  mean_test_score
44           XGBClassifier         0.827283
39           XGBClassifier         0.827263
34           XGBClassifier         0.824456
45           XGBClassifier         0.823047
42           XGBClassifier         0.820250
38           XGBClassifier         0.820240
10  RandomForestClassifier         0.818832
31           XGBClassifier         0.817423
9   RandomForestClassifier         0.816025
41           XGBClassifier         0.814626
35           XGBClassifier         0.814626
22  RandomForestClassifier         0.814616
6   RandomForestClassifier         0.814616
12  RandomForestClassifier         0.814597
40           XGBClassifier         0.814577
19  RandomForestClassifier         0.813198
3   RandomForestClassifier         0.813198
43           XGBClassifier         0.811819
4   RandomForestClassifier         0.811799
20  RandomForestClassifier         0.811799
8   RandomForestClassifier         0.811790
30           XGBClassifier      

/var/folders/v9/8v12n0k17v9586s8nm_r42t40000gn/T/ipykernel_24673/2081518084.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  summary['Model'] = summary['param_classifier'].apply(lambda x: type(x).__name__)


In [30]:
results = []

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000,random_state=42),
    "RandomForest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(eval_metric='logloss',random_state=42)
}

for name, clf in models.items():
    model = Pipeline([
        ('preprocessor', ColumnTransformer([
            ('num', Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())
            ]), ['age', 'fare']),
            ('cat', Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore'))
            ]), ['sex', 'embarked', 'pclass'])
        ])),
        ('classifier', clf)
    ])
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    test_acc = accuracy_score(y_test, y_pred)
    
    results.append({
        "Model": name,
        "CV Score": scores.mean(),
        "CV Std": scores.std(),
        "Test Score": test_acc
    })

import pandas as pd
df_results = pd.DataFrame(results)
print(df_results)

                Model  CV Score    CV Std  Test Score
0  LogisticRegression  0.793578  0.016486    0.770950
1        RandomForest  0.806146  0.017294    0.815642
2             XGBoost  0.803398  0.018039    0.804469
